# Data Quality Checks (Round 2)


In [8]:
import pandas as pd
from sqlalchemy import create_engine
import sys
sys.path.append('../src')
from config import SQLALCHEMY_URL

engine = create_engine(SQLALCHEMY_URL)
print('connected')

connected


In [9]:
customers = pd.read_sql('SELECT * FROM dim_customers', engine)
products = pd.read_sql('SELECT * FROM dim_products', engine)
sellers = pd.read_sql('SELECT * FROM dim_sellers', engine)
orders = pd.read_sql('SELECT * FROM dim_orders', engine)
order_items = pd.read_sql('SELECT * FROM fact_order_items', engine)
payments = pd.read_sql('SELECT * FROM fact_payments', engine)
reviews = pd.read_sql('SELECT * FROM fact_reviews', engine)

print('customers:', customers.shape)
print('products:', products.shape)
print('sellers:', sellers.shape)
print('orders:', orders.shape)
print('order_items:', order_items.shape)
print('payments:', payments.shape)
print('reviews:', reviews.shape)

customers: (99441, 5)
products: (32951, 6)
sellers: (3095, 4)
orders: (99441, 8)
order_items: (112650, 6)
payments: (103886, 5)
reviews: (98410, 5)


In [10]:
print('--- orders nulls ---')
print(orders.isnull().sum())

--- orders nulls ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [11]:
# Does every 'delivered' order actually have a delivery date?
delivered = orders[orders['order_status'] == 'delivered']
missing_delivery_date = delivered['order_delivered_customer_date'].isnull().sum()
print(f"Orders marked 'delivered' but missing a delivery date: {missing_delivery_date}")

Orders marked 'delivered' but missing a delivery date: 8


In [12]:
print(order_items['price'].describe())
print()
print('Items priced at 0 or less:', (order_items['price'] <= 0).sum())
print('Top 5 highest priced items:')
print(order_items.sort_values('price', ascending=False).head(5)[['product_id', 'price']])

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Items priced at 0 or less: 0
Top 5 highest priced items:
                              product_id   price
3556    489ae2aa008f021502940f251d4cce7f  6735.0
112233  69c590f7ffc7bf8db97190b6cb6ed62e  6729.0
107841  1bdf5e6731585cf01aa8169c7028d6ad  6499.0
74336   a6492cc69376c469ab6f61d8f44de961  4799.0
11249   c3ed642d592594bb648ff4a04cee2747  4690.0


In [13]:
dupe_check = customers.groupby('customer_unique_id')['customer_id'].nunique()
print('Customers with more than one customer_id:', (dupe_check > 1).sum())

Customers with more than one customer_id: 2997


In [14]:
orphan_items = ~order_items['order_id'].isin(orders['order_id'])
print('Order items with no matching order:', orphan_items.sum())

Order items with no matching order: 0
